# 01. Preflight & Baseline (Image × Regression, ADR-019)

02 본실행(~30분) 전 **빠른 sanity**: 데이터 로드 확인 + 베이스라인 DSC**+probe**가 정상인지.
결과 파일은 저장하지 않음 — baseline 포함 정식 산출은 02가 authoritative. SAMPLE_CAP은 02와 동일(2000).

In [ ]:
# 0-1. Drive 마운트 + GPU
from google.colab import drive
drive.mount('/content/drive')
import os, sys, gc
import numpy as np, pandas as pd, torch
BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'; DATA_DIR = f'{BASE}/data/image_regression'
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch {torch.__version__}')

In [ ]:
%pip install -q datasets timm imagehash opencv-python-headless

In [ ]:
# 사전등록 메타 (ADR-018/019)
DATASETS = {
    'UTKFace':      {'hf': 'Subh775/UTKFace_demographics_V1', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500': {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
SAMPLE_CAP = 2000; TEST_CAP = 2000; ML_SPLIT_SEED = 1; ML_TEST_SIZE = 0.2
print('datasets:', list(DATASETS.keys()), '| SAMPLE_CAP', SAMPLE_CAP)

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
def load_hf_split(ds_name):
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE, random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx
def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]; img = ex[meta['image_col']]
        if hasattr(img, 'convert'): img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8)); targets.append(float(ex[meta['target']]))
    return images, targets

In [ ]:
# import (DSC + 임베딩 + probe)
import importlib
if BASE not in sys.path: sys.path.insert(0, BASE)
importlib.invalidate_caches()
for _m in list(sys.modules):
    if _m.startswith('dsc_framework'): del sys.modules[_m]
from dsc_framework import compute_dsc_image_regression, DEFAULT_WEIGHTS_IMAGE_REG
from dsc_framework.image_cell import _extract_features
from dsc_framework.perf_probe import evaluate_probes
print('import OK | fallback 가중치 합', round(sum(DEFAULT_WEIGHTS_IMAGE_REG.values()), 2))

In [ ]:
# 베이스라인 preflight: clean train DSC + probe(train emb → clean test emb). 저장 X.
for ds_name in DATASETS:
    print(f'\n=== {ds_name} ===')
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    tr_img, tr_tgt = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    te_img, te_tgt = to_arrays(ds, meta, te_idx, sample_cap=TEST_CAP, random_state=1)
    del ds; gc.collect()
    res = compute_dsc_image_regression(tr_img, tr_tgt, sample_cap=SAMPLE_CAP); res.pop('metrics', None)
    feats_tr, idx_tr = _extract_features(tr_img, sample_cap=SAMPLE_CAP, random_state=1)
    feats_te, idx_te = _extract_features(te_img, sample_cap=TEST_CAP, random_state=1)
    y_tr = np.asarray(tr_tgt, dtype=float)[idx_tr]; y_te = np.asarray(te_tgt, dtype=float)[idx_te]
    probe = evaluate_probes(feats_tr, y_tr, feats_te, y_te, 'regression')
    print(f'  train {len(tr_img)} / test {len(te_img)}')
    print(f'  baseline DSC = {res["score"]} ({res["grade"]})  target_smoothness={res["target_smoothness"]}')
    print(f'  baseline probe R2 = { {k: v for k, v in probe.items() if not k.startswith("_")} }')
    del tr_img, tr_tgt, te_img, te_tgt, feats_tr, feats_te; gc.collect()
print('\n--- 01 preflight 완료. 정상이면 02 실행 ---')